In [1]:
import pandas as pd
import numpy as np
import scanpy as sc

GEARS

In [11]:
predictions = pd.read_csv('../../data/GEARS_predictions/mcfarland_mean_post_all.csv')

predictions = predictions.rename(columns={'test_condition': 'condition', 'cell_line': 'cell_type'})

predictions.to_csv('../../data/GEARS_predictions/mcfarland_mean_post_all.csv')


In [12]:
print(predictions.shape)
print(predictions.head())


(326, 19269)
   Unnamed: 0.1  Unnamed: 0    condition cell_type  fold      A1BG  \
0             0           0    MTOR+ctrl      786O     0  0.030950   
1             1           1    BCL2+ctrl      786O     0 -0.003347   
2             2           2    BRAF+ctrl      786O     1  0.013797   
3             3           3    BRD4+ctrl      786O     2 -0.017841   
4             4           4  MAP2K1+ctrl      786O     2 -0.016258   

           A1CF       A2M     A2ML1       A3GALT2  ...    ZWILCH     ZWINT  \
0 -4.138620e-07  0.000216  0.000012 -3.747337e-07  ...  0.273438  1.021613   
1 -4.076933e-07 -0.000681  0.000013 -3.729728e-07  ...  0.370309  1.195723   
2 -1.760271e-04  0.001031 -0.000747  5.612068e-04  ...  0.370896  1.168864   
3 -4.045235e-06 -0.000730 -0.000079  2.238155e-07  ...  0.459769  1.126679   
4 -4.043698e-06  0.001426 -0.000091  2.246671e-07  ...  0.481273  1.099470   

       ZXDA      ZXDB      ZXDC    ZYG11A    ZYG11B       ZYX     ZZEF1  \
0  0.033963 -0.019660 

In [13]:
pre_treatment = pd.read_csv('../../data/observed_pseudobulk/mcfarland_mean_pre_all_celllines.csv', index_col=0)

In [14]:
gene_cols = [col for col in pre_treatment.columns if col not in ['cell_type', 'condition', 'tissue']]

# Make sure combined_df has 'cell_type' column
assert 'cell_type' in predictions.columns

# Only keep gene columns from both dfs for subtraction
combined_gene_df = predictions[['cell_type'] + gene_cols].copy()
pre_gene_df = pre_treatment[['cell_type'] + gene_cols].copy()
pre_gene_df = pre_gene_df.drop_duplicates()

# Merge so that for each cell_type in combined_df, you have corresponding pre_treatment values
lfc_df = combined_gene_df.merge(pre_gene_df, on='cell_type', how='left', suffixes=('', '_pre'))

# Subtract pre-treatment gene values
for gene in gene_cols:
    lfc_df[gene] = lfc_df[gene] - lfc_df[gene + '_pre']

# Keep the LFC columns and cell_type
lfc_result = lfc_df[['cell_type'] + gene_cols]

# If you want to include other columns from combined_df (e.g. 'condition'), merge them in as needed
if 'condition' in predictions.columns:
    lfc_result['condition'] = predictions['condition']
if 'fold' in predictions.columns:
    lfc_result['fold'] = predictions['fold']

C:\Users\nbrouwer1\AppData\Local\Temp\ipykernel_43196\1200140340.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  lfc_result['condition'] = predictions['condition']
C:\Users\nbrouwer1\AppData\Local\Temp\ipykernel_43196\1200140340.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lfc_result['condition'] = predictions['condition']
C:\Users\nbrouwer1\AppData\Local\Temp\ipykernel_43196\1200140340.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti

In [15]:
print(lfc_result.shape)
print(lfc_result.head())

(326, 19267)
  cell_type      A1BG          A1CF       A2M     A2ML1       A3GALT2  \
0      786O  0.014699 -4.138620e-07 -0.000809  0.000012 -3.747337e-07   
1      786O -0.019598 -4.076933e-07 -0.001706  0.000013 -3.729728e-07   
2      786O -0.002454 -1.760271e-04  0.000006 -0.000747  5.612068e-04   
3      786O -0.034093 -4.045235e-06 -0.001755 -0.000079  2.238155e-07   
4      786O -0.032510 -4.043698e-06  0.000401 -0.000091  2.246671e-07   

     A4GALT         A4GNT      AAAS      AACS  ...      ZXDA      ZXDB  \
0 -0.056440  6.197743e-09  0.127514  0.065670  ...  0.021106 -0.035845   
1  0.032029 -2.662360e-10  0.073239  0.018536  ...  0.014238  0.024692   
2  0.031034 -3.556775e-06  0.049449 -0.013343  ...  0.002670 -0.011890   
3  0.032281  4.122337e-06  0.174086 -0.020774  ...  0.013578  0.032867   
4  0.053550  4.124710e-06  0.185780 -0.026730  ... -0.000372  0.018553   

       ZXDC    ZYG11A    ZYG11B       ZYX     ZZEF1      ZZZ3    condition  \
0 -0.021868 -0.000133 -0.

In [16]:
lfc_result.to_csv('../../data/GEARS_predictions/mcfarland_mean_LFC_all.csv', index=False)

CPA

In [46]:
adata = sc.read_h5ad(f'../../data/CPA_predictions/mcfarland_fold0.h5ad')


In [ ]:
adata

AnnData object with n_obs × n_vars = 196 × 19264
    obs: 'cell_type', 'condition', 'n_cells', 'fold'
    var: 'gene_name'
    layers: 'observed_mean', 'predicted_mean'

In [48]:
adatas = []
for i in range(0, 5):
    adata = sc.read_h5ad(f'../../data/CPA_predictions/mcfarland_fold{i}.h5ad')
    adatas.append(adata)

# Concatenate only the 'predicted_mean' layer matrices (assumed same variable order)
X_predicted = np.concatenate([a.layers["predicted_mean"] for a in adatas], axis=0)

# Create dataframe from combined predicted expression matrix with variable names as columns
expr_df = pd.DataFrame(X_predicted, columns=adatas[0].var_names)

# Concatenate all obs dataframes, reset index
obs_combined = pd.concat([a.obs for a in adatas], axis=0).reset_index(drop=True)

# Concatenate obs columns onto the expression dataframe
combined_df = pd.concat([expr_df, obs_combined.reset_index(drop=True)], axis=1)


In [49]:
combined_df['cell_type'] = combined_df['cell_type'].str.split('_').str[0]

In [50]:
print(combined_df.shape)
print(combined_df.head())

(991, 19268)
       A1BG      A1CF       A2M     A2ML1   A3GALT2    A4GALT     A4GNT  \
0  0.366484  0.000805  0.007192  0.010630 -0.000952  0.021795 -0.004867   
1  0.382318  0.000912  0.007356  0.010986 -0.001112  0.014284 -0.004609   
2  0.496376 -0.003765  0.009026  0.003874 -0.000886  0.144166 -0.005020   
3  0.499055 -0.003795  0.010059  0.004001 -0.001119  0.140248 -0.004847   
4  0.092001 -0.001114  0.006490  0.003736 -0.002396  0.154173 -0.009416   

       AAAS      AACS     AADAC  ...      ZXDC    ZYG11A    ZYG11B       ZYX  \
0  0.287363  0.236493 -0.007868  ...  0.066633  0.021505  0.179149  0.408325   
1  0.280561  0.236291 -0.007750  ...  0.064874  0.021192  0.177876  0.392859   
2  0.365395  0.118518 -0.004385  ...  0.064671  0.020406  0.133047  1.301345   
3  0.355447  0.120224 -0.004243  ...  0.063724  0.020207  0.133406  1.287099   
4  0.320890  0.194644 -0.009709  ...  0.057366  0.025614  0.171976  1.013641   

      ZZEF1      ZZZ3  cell_type  condition  n_cells  f

In [51]:
combined_df.to_csv('../../data/CPA_predictions/mcfarland_mean_post_all.csv', index=False)

In [52]:
pre_treatment = pd.read_csv('../../data/observed_pseudobulk/mcfarland_mean_pre_all_celllines.csv', index_col=0)

In [53]:
# To compute the LFC, we first need to:
# 1. Align combined_df and pre_treatment by cell_type and gene columns
# 2. Subtract pre_treatment's gene values from combined_df's gene values for each matching cell_type

# Merge the two dataframes on cell_type to align genes
# First, make sure 'cell_type' exists and is a column, not index, in both dataframes

# Drop columns from pre_treatment that are not gene columns (like 'condition', 'tissue' if present)
gene_cols = [col for col in pre_treatment.columns if col not in ['cell_type', 'condition', 'tissue']]

# Make sure combined_df has 'cell_type' column
assert 'cell_type' in combined_df.columns

# Only keep gene columns from both dfs for subtraction
combined_gene_df = combined_df[['cell_type'] + gene_cols].copy()
pre_gene_df = pre_treatment[['cell_type'] + gene_cols].copy()
pre_gene_df = pre_gene_df.drop_duplicates()

# Merge so that for each cell_type in combined_df, you have corresponding pre_treatment values
lfc_df = combined_gene_df.merge(pre_gene_df, on='cell_type', how='left', suffixes=('', '_pre'))

# Subtract pre-treatment gene values
for gene in gene_cols:
    lfc_df[gene] = lfc_df[gene] - lfc_df[gene + '_pre']

# Keep the LFC columns and cell_type
lfc_result = lfc_df[['cell_type'] + gene_cols]

# If you want to include other columns from combined_df (e.g. 'condition'), merge them in as needed
if 'condition' in combined_df.columns:
    lfc_result['condition'] = combined_df['condition']
if 'fold' in combined_df.columns:
    lfc_result['fold'] = combined_df['fold']

C:\Users\nbrouwer1\AppData\Local\Temp\ipykernel_21288\43137206.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  lfc_result['condition'] = combined_df['condition']
C:\Users\nbrouwer1\AppData\Local\Temp\ipykernel_21288\43137206.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lfc_result['condition'] = combined_df['condition']
C:\Users\nbrouwer1\AppData\Local\Temp\ipykernel_21288\43137206.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w

In [54]:
print(lfc_result.shape)
print(lfc_result.head())

(991, 19267)
  cell_type      A1BG      A1CF       A2M     A2ML1   A3GALT2    A4GALT  \
0     22RV1  0.128746 -0.019519 -0.008084  0.010630 -0.000952 -0.027412   
1     22RV1  0.144580 -0.019412 -0.007920  0.010986 -0.001112 -0.034923   
2    42MGBA -0.383271 -0.007696  0.009026  0.003874 -0.000886  0.116847   
3    42MGBA -0.380592 -0.007726  0.010059  0.004001 -0.001119  0.112928   
4      769P  0.040166 -0.021125 -0.000332  0.003736 -0.002396 -0.087257   

      A4GNT      AAAS      AACS  ...      ZXDA      ZXDB      ZXDC    ZYG11A  \
0 -0.004867 -0.039028 -0.005275  ...  0.015836  0.016136 -0.089732 -0.003297   
1 -0.004609 -0.045830 -0.005477  ...  0.017061  0.016639 -0.091490 -0.003610   
2 -0.005020  0.074930 -0.021094  ...  0.006624  0.052507 -0.044296  0.013900   
3 -0.004847  0.064981 -0.019388  ...  0.007405  0.053617 -0.045243  0.013702   
4 -0.009416  0.068059 -0.012770  ... -0.028259  0.000261 -0.000215 -0.001833   

     ZYG11B       ZYX     ZZEF1      ZZZ3  condition  f

In [55]:
lfc_result.to_csv('../../data/CPA_predictions/mcfarland_mean_LFC_all.csv', index=False)

scfoundation

In [4]:
predictions = pd.read_csv('../../data/scfoundation_predictions/all_cell_lines_test_pseudobulk.csv')

predictions = predictions.rename(columns={'test_condition': 'condition', 'cell_line': 'cell_type'})
predictions.drop(columns=['drug'], inplace=True)

predictions.to_csv('../../data/scfoundation_predictions/mcfarland_mean_post_all.csv')


In [5]:
print(predictions.shape)
print(predictions.head())


(365, 19267)
   condition cell_type  fold      A1BG          A1CF       A2M         A2ML1  \
0  MTOR+ctrl      786O     0  0.126470 -1.362340e-39  0.000030  1.197680e-40   
1  BCL2+ctrl      786O     0  0.133766 -1.354260e-39  0.002182  1.095340e-40   
2  BRAF+ctrl      786O     1  0.106702 -1.314330e-40  0.040652  3.445850e-40   
3  EGFR+ctrl      786O     1  0.104880 -1.297620e-40  0.040703  3.515600e-40   
4  BRD4+ctrl      786O     2  0.128945  2.712960e-40  0.000672  5.825664e-10   

        A3GALT2    A4GALT         A4GNT  ...    ZWILCH     ZWINT      ZXDA  \
0  1.170150e-39  0.106613 -1.188581e-39  ...  0.390132  0.903773 -0.019102   
1  1.140857e-39  0.082092 -1.181188e-39  ...  0.339215  0.958560 -0.010009   
2 -4.312410e-40  0.084653 -2.071160e-40  ...  0.374000  0.892491  0.029333   
3 -4.299980e-40  0.061033 -2.074060e-40  ...  0.321640  0.958864  0.037192   
4  3.529320e-40  0.049701  3.983200e-40  ...  0.354338  1.050061  0.056559   

       ZXDB      ZXDC    ZYG11A    ZY

In [6]:
pre_treatment = pd.read_csv('../../data/observed_pseudobulk/mcfarland_mean_pre_all_celllines.csv', index_col=0)

In [7]:
gene_cols = [col for col in pre_treatment.columns if col not in ['cell_type', 'condition', 'tissue']]

# Make sure combined_df has 'cell_type' column
assert 'cell_type' in predictions.columns

# Only keep gene columns from both dfs for subtraction
combined_gene_df = predictions[['cell_type'] + gene_cols].copy()
pre_gene_df = pre_treatment[['cell_type'] + gene_cols].copy()
pre_gene_df = pre_gene_df.drop_duplicates()

# Merge so that for each cell_type in combined_df, you have corresponding pre_treatment values
lfc_df = combined_gene_df.merge(pre_gene_df, on='cell_type', how='left', suffixes=('', '_pre'))

# Subtract pre-treatment gene values
for gene in gene_cols:
    lfc_df[gene] = lfc_df[gene] - lfc_df[gene + '_pre']

# Keep the LFC columns and cell_type
lfc_result = lfc_df[['cell_type'] + gene_cols]

# If you want to include other columns from combined_df (e.g. 'condition'), merge them in as needed
if 'condition' in predictions.columns:
    lfc_result['condition'] = predictions['condition']
if 'fold' in predictions.columns:
    lfc_result['fold'] = predictions['fold']

C:\Users\nbrouwer1\AppData\Local\Temp\ipykernel_43196\1200140340.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  lfc_result['condition'] = predictions['condition']
C:\Users\nbrouwer1\AppData\Local\Temp\ipykernel_43196\1200140340.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lfc_result['condition'] = predictions['condition']
C:\Users\nbrouwer1\AppData\Local\Temp\ipykernel_43196\1200140340.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many ti

In [8]:
print(lfc_result.shape)
print(lfc_result.head())

(365, 19267)
  cell_type      A1BG          A1CF       A2M         A2ML1       A3GALT2  \
0      786O  0.110219 -1.362340e-39 -0.000995  1.197680e-40  1.170150e-39   
1      786O  0.117514 -1.354260e-39  0.001157  1.095340e-40  1.140857e-39   
2      786O  0.090450 -1.314330e-40  0.039628  3.445850e-40 -4.312410e-40   
3      786O  0.088628 -1.297620e-40  0.039678  3.515600e-40 -4.299980e-40   
4      786O  0.112694  2.712960e-40 -0.000353  5.825664e-10  3.529320e-40   

     A4GALT         A4GNT      AAAS      AACS  ...      ZXDA      ZXDB  \
0  0.021706 -1.188581e-39 -0.112205 -0.121057  ... -0.031959  0.036876   
1 -0.002815 -1.181188e-39 -0.100035 -0.098261  ... -0.022867  0.032183   
2 -0.000254 -2.071160e-40  0.095482  0.037980  ...  0.016476  0.048198   
3 -0.023874 -2.074060e-40  0.091091  0.041837  ...  0.024335  0.054695   
4 -0.035206  3.983200e-40 -0.058642  0.055528  ...  0.043701  0.007138   

       ZXDC    ZYG11A    ZYG11B       ZYX     ZZEF1      ZZZ3  condition  fold 

In [10]:
lfc_result.to_csv('../../data/scfoundation_predictions/mcfarland_mean_LFC_all.csv', index=False)